# Final report and claim traceability

**Purpose:** Display the public final report together with the claim registry evidence table for traceability.

**Inputs (committed result/provenance files read):**
- [`results/reports/FINAL_REPORT.md`](../results/reports/FINAL_REPORT.md)
- [`publication/CLAIM_LEDGER.md`](../publication/CLAIM_LEDGER.md)
- [`publication/claim_evidence.csv`](../publication/claim_evidence.csv)

**Outputs:** Rendered report excerpt and claim-evidence table in this notebook only; no files are written.

**Expected runtime:** ~3 seconds (loads committed CSV/JSON/parquet files only).

**Traceability links:** [`FINAL_REPORT.md`](../results/reports/FINAL_REPORT.md) and [`CLAIM_LEDGER.md`](../publication/CLAIM_LEDGER.md).

**Execution notes:** Analysis-only notebook. It resolves `ROOT` by walking upward to `plan.md`, uses no absolute paths or machine identifiers, does not download corpora, does not retrain tokenizers, and has no hidden state. Randomness: none used. Exploratory analysis: none; cells below directly present committed result files.


In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / 'plan.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

def rel(path):
    path = Path(path)
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)

def note_missing(path):
    print(f'NOTE: optional input is missing or empty: {path}. The notebook continues without it.')

def load_csv(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return pd.DataFrame()
    return pd.read_csv(p)

def load_parquet(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return pd.DataFrame()
    try:
        return pd.read_parquet(p)
    except Exception as exc:
        print(f'NOTE: could not read optional parquet {path}: {exc}')
        return pd.DataFrame()

def load_json(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return {}
    return json.loads(p.read_text())

def load_text(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return ''
    return p.read_text()

def safe_read_json_path(path):
    try:
        return json.loads(Path(path).read_text())
    except Exception as exc:
        print(f'NOTE: could not read {rel(path)}: {exc}')
        return {}

print(f'Repository root: {rel(ROOT)}')


Repository root: .


## Final report and claim registry

Authoritative files: [`FINAL_REPORT.md`](../results/reports/FINAL_REPORT.md), [`CLAIM_LEDGER.md`](../publication/CLAIM_LEDGER.md), and [`claim_evidence.csv`](../publication/claim_evidence.csv).

In [2]:
final_report = load_text('results/reports/FINAL_REPORT.md')
claim_ledger = load_text('publication/CLAIM_LEDGER.md')
claim_evidence = load_csv('publication/claim_evidence.csv')

In [3]:
if final_report:
    print(final_report[:5000])
else:
    note_missing('results/reports/FINAL_REPORT.md')

# Does Your Tokenizer Speak Cloud?
## Measuring domain specialization on Microsoft technical documentation

_Profile: **full**. This report is generated from committed result tables; every figure has a source CSV under `results/tables/`._

> **Scope.** This study measures **tokenization only** — token counts, compression, fragmentation, vocabulary utilization, and CPU runtime. It makes **no claim** about model output quality, accuracy, reasoning, understanding, GPU latency, or cost (plan.md §4, §26).

## 1. Research question
At a fixed tokenizer algorithm, vocabulary size, training-data size, normalization policy, and evaluation set, does training a byte-level BPE tokenizer on Azure documentation reduce tokens-per-1000-UTF-8-bytes on unseen technical documentation versus training on byte-matched general English?

## 2. Hypotheses
- **Primary:** Azure-trained tokenizer uses fewer tokens/1000 bytes on the Azure cross-service holdout than a byte-matched general-English tokenizer.
- **Seco

In [4]:
if not claim_evidence.empty:
    print(f'Claim evidence rows: {len(claim_evidence):,}')
    display(claim_evidence)
else:
    note_missing('publication/claim_evidence.csv')

Claim evidence rows: 43


,id,type,status,display,actual,tolerance,source
0,PRIMARY_REDUCTION_16K,primary,OK,36.68,36.6772823913683,0.050,results/processed/bootstrap.json
1,PRIMARY_CI_LOW,primary,OK,35.77,35.767852757203215,0.050,results/processed/bootstrap.json
2,PRIMARY_CI_HIGH,primary,OK,37.62,37.61822932254951,0.050,results/processed/bootstrap.json
3,PRIMARY_WIN_RATE,primary,OK,1.0,1.0,0.001,results/processed/bootstrap.json
4,PRIMARY_N_DOCS,primary,OK,964,964,0.000,results/processed/bootstrap.json
5,PRIMARY_AZURE_TPK,descriptive,OK,240.18,240.18021302918405,0.020,results/processed/eval_aggregate.csv
6,PRIMARY_GENERAL_TPK,descriptive,OK,379.3,379.2954915700655,0.020,results/processed/eval_aggregate.csv
7,PRIMARY_MIXED_TPK,descriptive,OK,249.52,249.52310090655277,0.020,results/processed/eval_aggregate.csv
8,MIXED_REDUCTION_PRINCIPAL,primary,OK,34.21,34.21406094924291,0.050,results/processed/bootstrap.json
9,GENERAL_REGRESSION_AZURE,primary,OK,-34.75,-34.74744696153846,0.050,results/processed/bootstrap.json


In [5]:
if claim_ledger:
    print(claim_ledger[:3000])
else:
    note_missing('publication/CLAIM_LEDGER.md')

# Claim Ledger

> Human-readable index of the programmatic claim registry. The machine-readable
> source of truth is [`claims.yaml`](claims.yaml); resolved values are in
> [`claim_evidence.csv`](claim_evidence.csv). Regenerate/verify with
> `make verify-claims` (`uv run python scripts/verify_claims.py`).

- **Total registered claims:** 39
- **Last verification:** 39/39 PASS
- **Rule:** No number may appear in README, docs, figures, tables, captions,
  social copy, release notes, or the paper unless it is registered here and
  passes verification.

## Headline claims (all reconciled to `results/`)

| Claim ID | Value | Source file | Key/column |
|---|---:|---|---|
| PRIMARY_REDUCTION_16K | 36.68% | results/processed/bootstrap.json | comparisons.azure_vs_general:16384:azure_cross_service_holdout.point_estimate |
| PRIMARY_CI_LOW / HIGH | 35.77% / 37.62% | results/processed/bootstrap.json | …ci_low / ci_high |
| PRIMARY_WIN_RATE | 1.000 | results/processed/bootstrap.json | …win_rate |
| P